In [14]:
import luadata
from pprint import pp
from mgrs import MGRS
from pyproj import Transformer, CRS
import csv

In [15]:
ORIGIN_MGRS = "34W EA 62702 43625"

In [16]:
data = luadata.read(r'Kola.lua', encoding='utf-8')

In [17]:
pp(data['Radars2'])

[{'speed_locked': False,
  'type': 'Turning Point',
  'action': 'Turning Point',
  'ETA_locked': True,
  'y': 332102.0625,
  'x': 202541.640625,
  'name': 'sa2',
  'ETA': 900,
  'alt_type': 'RADIO',
  'alt': 0},
 {'alt': 0,
  'type': 'Turning Point',
  'ETA': 1925.5257216623,
  'ETA_locked': True,
  'y': 392221.5625,
  'x': 163311.671875,
  'name': 'EWR',
  'action': 'Turning Point',
  'alt_type': 'RADIO',
  'speed_locked': False},
 {'alt': 0,
  'type': 'Turning Point',
  'ETA': 2572.3448328448,
  'ETA_locked': True,
  'y': 436743.75,
  'x': 155076.859375,
  'name': 'SA5',
  'action': 'Turning Point',
  'alt_type': 'RADIO',
  'speed_locked': False},
 {'alt': 0,
  'type': 'Turning Point',
  'ETA': 2663.1068101186,
  'ETA_locked': True,
  'y': 443092.1875,
  'x': 155326.359375,
  'name': 'SA5',
  'action': 'Turning Point',
  'alt_type': 'RADIO',
  'speed_locked': False},
 {'speed_locked': False,
  'type': 'Turning Point',
  'action': 'Turning Point',
  'ETA_locked': True,
  'y': 350841.9

In [18]:
# import sys
# import subprocess

# convert an MGRS string, add east/north offsets (meters) in the local planar grid,
# and return the resulting lat/lon and MGRS coordinate.
# (Requires mgrs and pyproj; this cell will install them if missing.)


# def _ensure(pkg):
#     try:
#         __import__(pkg)
#     except Exception:
#         subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# _ensure("mgrs")
# _ensure("pyproj")


# Input
def convert_offset_to_coords(origin_mgrs, north_offset, east_offset):
    # origin_mgrs = "34W EA 62702 43625"
    # north_offset = 202541  # meters to add (north)
    # east_offset = 332102   # meters to add (east)

    # normalize MGRS string (mgrs library accepts compact form)
    mgrs_str = origin_mgrs.replace(" ", "")

    m = MGRS()

    # convert MGRS -> lat, lon
    # mgrs.toLatLon sometimes returns (lat, lon)
    lat_lon = m.toLatLon(mgrs_str)
    if isinstance(lat_lon, (bytes, bytearray)):
        # unlikely, but decode if needed
        lat_lon = lat_lon.decode()

    # ensure we have floats (some mgrs versions return tuple)
    if isinstance(lat_lon, tuple) and len(lat_lon) == 2:
        lat_origin, lon_origin = float(lat_lon[0]), float(lat_lon[1])
    else:
        raise RuntimeError("Unexpected return from mgrs.toLatLon")

    # build an Azimuthal Equidistant projection centered on the origin.
    # In this projection, x ~ east (meters), y ~ north (meters).
    # aeqd_proj = CRS.from_proj4(f"+proj=tmerc +lat_0={lat_origin} +lon_0={lon_origin} +datum=WGS84 +units=m +no_defs +k_0=1")
    # aeqd_proj = CRS.from_proj4(f"+proj=tmerc +lat_0={lat_origin} +lon_0={lon_origin} +datum=WGS84 +units=m +no_defs +k_0=0.9996")
    # aeqd_proj = CRS.from_proj4(f"+proj=aeqd +lat_0={lat_origin} +lon_0={lon_origin} +datum=WGS84 +units=m +no_defs")
    
    aeqd_proj = CRS.from_proj4(
            " ".join(
                [
                    "+proj=tmerc",
                    "+lat_0=0",
                    f"+lon_0=21",
                    f"+k_0=0.9996",
                    f"+x_0={lon_origin}",
                    f"+y_0={lat_origin}",
                    "+towgs84=0,0,0,0,0,0,0",
                    "+units=m",
                    "+vunits=m",
                    "+ellps=WGS84",
                    "+no_defs",
                    "+axis=neu",
                ]
            ))

    to_aeqd = Transformer.from_crs(CRS("WGS84"), aeqd_proj, always_xy=True)   # input is lon,lat
    from_aeqd = Transformer.from_crs(aeqd_proj, CRS("WGS84"), always_xy=True) # output is lon,lat

    # origin in AEQD coordinates (should be near 0,0)
    x0, y0 = to_aeqd.transform(lon_origin, lat_origin)

    # add offsets (east -> +x, north -> +y)
    x_new = x0 + (east_offset)
    y_new = y0 + (north_offset)
    # x_new = x0 + (east_offset * 0.9996)
    # y_new = y0 + (north_offset * 0.9996)

    # transform back to geographic coordinates
    lon_new, lat_new = from_aeqd.transform(x_new, y_new)

    # convert result back to MGRS. Use precision 5 to match 1-meter digits (same as input).
    # mgrs.toMGRS(lat, lon, precision) may return bytes on some installs, handle that.
    mgrs_result = m.toMGRS(lat_new, lon_new, MGRSPrecision=5)
    if isinstance(mgrs_result, (bytes, bytearray)):
        mgrs_result = mgrs_result.decode()

    # pretty-format MGRS (insert spaces similar to original: zone(3) + 2 letters + easting + northing)
    zone = mgrs_result[:3]
    letters = mgrs_result[3:5]
    digits = mgrs_result[5:]
    half = len(digits) // 2
    easting = digits[:half]
    northing = digits[half:]
    pretty_mgrs = f"{zone} {letters} {easting} {northing}"

    # print("Origin MGRS:", origin_mgrs)
    # print("Origin lat, lon:", lat_origin, lon_origin)
    # print("Offsets (north, east) m:", north_offset, east_offset)
    # print("New lat, lon:", lat_new, lon_new)
    # print("New MGRS (compact):", mgrs_result)
    # print("New MGRS (spaced):", pretty_mgrs)
    return lat_new, lon_new, mgrs_result, pretty_mgrs

In [19]:
convert_offset_to_coords('34W EA 62702 43625', 202541, 332102)

(69.52838508162365, 31.15499344347183, '36WVC2799414367', '36W VC 27994 14367')

In [20]:
def trunc_round(num, precision=3):
    parts = (str(num).split('.'))
    if len(parts) < 2:
        return (str(num) + '.' + ('0' * precision)) # no decimal part
    
    integer = parts[0]
    decimal = parts[1][:precision]
    if len(decimal) < precision:
        decimal = decimal + ('0' * (precision - len(decimal)))
    return (integer + '.' + decimal)

# print(trunc_round(lat_dms[2], 6), trunc_round(lon_dms[2], 6))
# print(trunc_round(60, 6), trunc_round(60.0, 6))

In [21]:
def latlon_to_dms(lat, lon):
    '''Converts lat_new to DMS precise format (degrees, minutes, seconds)'''
    def to_dms(value):
        degrees = int(value)
        minutes_full = abs((value - degrees) * 60)
        minutes = int(minutes_full)
        seconds = (minutes_full - minutes) * 60
        return degrees, minutes, seconds

    lat_deg, lat_min, lat_sec = to_dms(lat)
    lon_deg, lon_min, lon_sec = to_dms(lon)

    return (f"{lat_deg}°{lat_min}'{trunc_round(lat_sec, 2)}"), (f"{lon_deg}°{lon_min}'{trunc_round(lon_sec, 2)}")

lat_dms, lon_dms = latlon_to_dms(67.99999682172047, 22.499998619459987)
print("New lat DMS:", lat_dms)
print("New lon DMS:", lon_dms)

New lat DMS: 67°59'59.98
New lon DMS: 22°29'59.99


In [22]:
def latlon_to_decimal_minutes(lat, lon):
    '''Converts lat_new to Decimal Minutes format'''
    def to_decimal_minutes(value):
        degrees = int(value)
        minutes_full = abs((value - degrees) * 60)
        return degrees, minutes_full

    lat_deg, lat_min = to_decimal_minutes(lat)
    lon_deg, lon_min = to_decimal_minutes(lon)

    return (f'{lat_deg}°{trunc_round(lat_min,3)}'), (f'{lon_deg}°{trunc_round(lon_min,3)}')

lat_dm, lon_dm = latlon_to_decimal_minutes(67.99999682172047, 22.499998619459987)
print("New lat Decimal Minutes:", lat_dm)
print("New lon Decimal Minutes:", lon_dm)

New lat Decimal Minutes: 67°59.999
New lon Decimal Minutes: 22°29.999


In [23]:
def build_radar_coords(radar):
    radar_coords = {}
    
    name = radar['name'].upper()
    north_offset = radar['x']
    east_offset = radar['y']
    lat_new, lon_new, mgrs_result, pretty_mgrs = convert_offset_to_coords(ORIGIN_MGRS, north_offset, east_offset)
    
    lat_dms, lon_dms = latlon_to_dms(lat_new, lon_new)
    lat_dm, lon_dm = latlon_to_decimal_minutes(lat_new, lon_new)
    
    
    radar_coords = {
        'name': name,
        'north_offset': north_offset,
        'east_offset': east_offset,
        'latitude': lat_new,
        'longitude': lon_new,
        'lat_prcise': lat_dms,
        'lon_prcise': lon_dms,
        'lat_dm': lat_dm,
        'lon_dm': lon_dm,
        'mgrs_compact': mgrs_result,
        'mgrs_pretty': pretty_mgrs
    }
    return radar_coords

In [24]:
radars = []
for i, radar in enumerate(data['Radars2']):
    
    info = build_radar_coords(radar)
    
    radars.append((
        i+1,
        info['name'],
        info['north_offset'],
        info['east_offset'],
        info['latitude'],
        info['longitude'],
        info['lat_prcise'],
        info['lon_prcise'],
        info['lat_dm'],
        info['lon_dm'],
        info['mgrs_compact'],
        info['mgrs_pretty']
    ))

In [25]:
radars

[(1,
  'SA2',
  202541.640625,
  332102.0625,
  69.52839064396765,
  31.15499773059966,
  "69°31'42.20",
  "31°9'17.99",
  '69°31.703',
  '31°9.299',
  '36WVC2799414368',
  '36W VC 27994 14368'),
 (2,
  'EWR',
  163311.671875,
  392221.5625,
  69.08815723223452,
  32.477859055865956,
  "69°5'17.36",
  "32°28'40.29",
  '69°5.289',
  '32°28.671',
  '36WVB7920164280',
  '36W VB 79201 64280'),
 (3,
  'SA5',
  155076.859375,
  436743.75,
  68.93838909315227,
  33.52739994575486,
  "68°56'18.20",
  "33°31'38.63",
  '68°56.303',
  '33°31.643',
  '36WWB2115147583',
  '36W WB 21151 47583'),
 (4,
  'SA5',
  155326.359375,
  443092.1875,
  68.92897809846244,
  33.68310885812138,
  "68°55'44.32",
  "33°40'59.19",
  '68°55.738',
  '33°40.986',
  '36WWB2740846595',
  '36W WB 27408 46595'),
 (5,
  'SA3',
  238458.390625,
  350841.96875,
  69.81647374480968,
  31.788203999385345,
  "69°48'59.30",
  "31°47'17.53",
  '69°48.988',
  '31°47.292',
  '36WVC5334045870',
  '36W VC 53340 45870'),
 (6,
  'SA2',

In [26]:
# Export radars to CSV with headers
with open('radar_coords.csv', mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    # Write header
    writer.writerow([
        'Index',
        'name',
        'north_offset',
        'east_offset',
        'latitude',
        'longitude',
        'lat_prcise',
        'lon_prcise',
        'lat_dm',
        'lon_dm',
        'mgrs_compact',
        'mgrs_pretty'
    ])
    # Write radar data
    for radar in radars:
        writer.writerow(radar)